### 실습 환경 준비

In [2]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnableParallel, RunnablePassthrough
from langchain_core.documents import Document
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader
from transformers import AutoTokenizer
from pydantic import BaseModel, Field
from dotenv import load_dotenv
from enum import Enum
from typing import Any


load_dotenv("../.env")

True

In [3]:
embeddings = HuggingFaceEmbeddings(
    model_name="intfloat/multilingual-e5-base",
    model_kwargs={"device": "cuda"},
    encode_kwargs={"prompt": "passage: ", "normalize_embeddings": True},
    query_encode_kwargs={"prompt": "query: ", "normalize_embeddings": True}
)

db = Chroma(
    persist_directory="../ch06_advanced_rag/chroma_db_e5",
    collection_name="langchain-docs",
    embedding_function=embeddings,
    create_collection_if_not_exists=False
)

Loading weights: 100%|██████████| 199/199 [00:00<00:00, 2359.53it/s]


In [9]:
loader = DirectoryLoader(
    path="../ch06_advanced_rag/langchain-docs",
    glob="**/*.mdx",
    loader_cls=TextLoader,
    loader_kwargs={"encoding": "utf-8"},
    use_multithreading=True,
)

documents = loader.load()
print(f"{len(documents)}")

2532


In [5]:
tokenizer = AutoTokenizer.from_pretrained("intfloat/multilingual-e5-base")
text_splitter = RecursiveCharacterTextSplitter.from_huggingface_tokenizer(
    tokenizer=tokenizer,
    chunk_size=400,
    chunk_overlap=50,
    separators=["\n\n", "\n", " ", ""]
)

chunks = text_splitter.split_documents(documents)
print(f"{len(chunks)}")

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (4739 > 512). Running this sequence through the model will result in indexing errors


18413


### Ragas를 활용한 합성 테스트 데이터 생성 구현

In [14]:
import nest_asyncio
from openai import AsyncOpenAI
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings as RagasHuggingFaceEmbeddings
from ragas.testset import TestsetGenerator

nest_asyncio.apply()

generator_embeddings = RagasHuggingFaceEmbeddings(
    model="intfloat/multilingual-e5-base",
    device="cuda",
    normalize_embeddings=True,
    prompts={"passage": "passage: "},
    default_prompt_name="passage"
)

client = AsyncOpenAI()
generator_llm = llm_factory(
    model="gpt-6-luna",
    client=client,
    reasoning_effort="none",
    max_tokens=4096,
)

generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings
)

testset = generator.generate_with_langchain_docs(
    documents=documents[:6],
    testset_size=6
)

testset_df = testset.to_pandas()
print(f"생성 개수: {len(testset_df)}")
testset_df

Applying CustomNodeFilter:   5%|▍         | 1/21 [00:02<00:40,  2.00s/it]Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://aws.api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://aws.api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a19ae0-a3f8-75c5-915a-73a174c53cee,id=01a19ae0-a3f8-75c5-915a-73a174c53cee; trace=01a19ae0-a3fa-71c1-a3df-13d6b0f65f82,id=01a19ae0-a3fa-71c1-a3df-13d6b0f65f82; trace=01a19ae0-a400-7377-a820-a41f38e5ac58,id=01a19ae0-a400-7377-a820-a41f38e5ac58; trace=01a19ae0-c51f-76e8-9db2-fb9fdff1f206,id=01a19ae0-c51f-76e8-9db2-fb9fdff1f206; trace=01a19ae0-c523-777d-8f97-cfbf0e80e03d,id=01a19ae0-c523-777d-8f97-cfbf0e80e03d; trace=01a19ae0-c528-77f7-ae04-c26b1c9041a1,id=01a19ae0-c528-77f7-ae04-c26b1c9041a1; trace=01a19ae0-c52b-71cd-bb00-736cb6a255eb,id

생성 개수: 6


,user_input,reference_contexts,reference,persona_name,query_style,query_length,synthesizer_name
0,How LangChain help build agents?,[---\ntitle: Build\nsidebarTitle: Overview\nmo...,The LangChain open source stack provides build...,LangSmith Enterprise Access Administrator,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
1,How Deep Agents give me more control?,[Choose your starting point</h2>\n\n Ma...,Deep Agents use the same harness as Managed De...,LangSmith Enterprise Access Administrator,POOR_GRAMMAR,SHORT,single_hop_specific_query_synthesizer
2,How can resource tags support ABAC policies th...,[<1-hop>\n\n---\ntitle: Attribute-based access...,ABAC evaluates resource tags to control access...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
3,How does resource tag-based access control com...,[<1-hop>\n\n---\ntitle: Attribute-based access...,LangSmith ABAC evaluates access using resource...,NaN,NaN,NaN,multi_hop_abstract_query_synthesizer
4,For a Deep agent workflow handling a long-runn...,[<1-hop>\n\ncheckpoint using LangChain state. ...,The Deep agent — Subagent Cards pattern presen...,NaN,NaN,NaN,multi_hop_specific_query_synthesizer
5,I am troubleshooting access and tracing in a L...,[<1-hop>\n\nAccess the current run (span) with...,"Within a traced function, call `get_current_ru...",NaN,NaN,NaN,multi_hop_specific_query_synthesizer


Failed to send compressed multipart ingest: langsmith.utils.LangSmithRateLimitError: Rate limit exceeded for https://aws.api.smith.langchain.com/runs/multipart. HTTPError('429 Client Error: Too Many Requests for url: https://aws.api.smith.langchain.com/runs/multipart', '{"error":"Too many requests: tenant exceeded usage limits: Monthly unique traces usage limit exceeded"}\n')trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f7b-70cb-81fb-6d63b649dedb; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f82-72dd-a283-61d620dee9f2; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f81-7057-b7fd-23b3e0422af7; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f83-7266-aedb-cd9397db26b9; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f83-7266-aedb-cd92fb262bcb; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f85-77a6-b877-cbef981a1a8f; trace=01a19ae1-e6b7-77d0-a6b2-c31b46807f36,id=01a19ae2-2f85-77a6-b877-cbee71477b7a; trace=01a19ae1-e6b7-77d0-a6b2-c31b

### LangSmith의 Dataset 생성

In [22]:
from langsmith import Client

dataset_name = "agent-book"
langsmith_client = Client()

if langsmith_client.has_dataset(dataset_name=dataset_name):
    langsmith_client.delete_dataset(dataset_name=dataset_name)

dataset = langsmith_client.create_dataset(dataset_name=dataset_name)

### 합성 테스트 데이터 저장

In [17]:
# Ragas로 생성한 데이터를 LangSmith의 Dataset에 저장하는 코드를 작성하고 실행해 보자
# 먼저 생성한 데이터셋을 LangSmith의 Dataset에 저장하는 형식으로 변환

records = testset.to_list()

inputs = []
outputs = []
metadatas = []

for record in records:
    inputs.append({
        "question": record["user_input"]
    })
    outputs.append({
        "answer": record["reference"],
        "context": record["reference_contexts"]
    })
    metadatas.append({
        "synthesize_name": record["synthesizer_name"]
    })

In [23]:
# 이어서 LangSmith 클라이언트를 사용해 Dataset의 ID를 지정하고 데이터를 저장
# LangSmith에서는 Dataset에 저장하는 데이터의 각 항목을 Example이라고 부른다.
langsmith_client.create_examples(
    inputs=inputs,
    reference_outputs=outputs,
    metadata=metadatas,
    dataset_id=dataset.id
)

{'example_ids': ['3ba6dab4-5e96-459c-8660-bee1677303e0',
  '87fba64d-1dba-461c-b341-6018d75e7621',
  '58cc5336-b599-4e35-97df-890fe0a1615c',
  '06ea9a23-fbf6-42d8-bf6a-b4c934b09269',
  '21ba117b-7455-4df0-8a04-b55f896c44af',
  '3d2b82e8-4652-4044-95bd-7590e7f53c39'],
 'count': 6,
 'as_of': '2026-11-02T06:21:47.443882231Z'}